# DRKG → Knowledge Graph (KG) Builder

**Source:** DRKG (Drug Repurposing Knowledge Graph) — `drkg.tsv`  
**Species:** *Homo sapiens* (non-human gene entries filtered out)

## What this notebook does

**Part 1 — Split:** Loads `drkg.tsv`, parses the `EntityType::ID` format, builds per-relation CSVs saved in `DRKG_relation_wise/`.

**Part 2 — Process:** For each of the **13 active relation types**, normalises Head/Tail IDs to canonical ontology identifiers and exports one CSV.

## Relation types processed (13 active)

| Relation | Head ID | Tail ID |
|---|---|---|
| Gene_Disease | NCBI GeneID → Symbol | DOID / MeSH ID |
| Disease_Anatomy | DOID / MeSH ID | UBERON ID |
| Gene_Biological_Process | NCBI GeneID → Symbol | GO ID |
| Disease_Disease | DOID / MeSH ID | DOID / MeSH ID |
| Compound_Disease | DrugBank/ChEBI/MeSH → PubChem CID | DOID / MeSH ID |
| Disease_Gene | DOID / MeSH ID | NCBI GeneID → Symbol |
| Gene_Compound | NCBI GeneID → Symbol | DrugBank/ChEBI/SMILES → PubChem CID |
| Gene_Cellular_Component | NCBI GeneID → Symbol | GO ID |
| Gene_Gene | NCBI GeneID → Symbol | NCBI GeneID → Symbol |
| Compound_Compound | DrugBank → PubChem CID | DrugBank → PubChem CID |
| Anatomy_Gene | UBERON ID | NCBI GeneID → Symbol |
| Gene_Molecular_Function | NCBI GeneID → Symbol | GO ID |
| Compound_Gene | DrugBank/ChEBI/SMILES → PubChem CID | NCBI GeneID → Symbol |
| Compound_Side_Effect | DrugBank → PubChem CID | UMLS CUI → HPO ID |

#### Dropped (intentionally excluded)
- `Pharmacologic_Class_Compound` — entity IDs not available in standard ontologies
- `Compound_Atc` — ATC classification codes not in scope
- `Gene_Tax` — taxonomy edges not in scope
- `Disease_Symptom` — processing logic incomplete in original (all cells commented out)

## Output files
```
DRKG/   DRKG_Gene_Disease.csv          DRKG/   DRKG_Disease_Gene.csv
DRKG/   DRKG_Disease_Anatomy.csv       DRKG/   DRKG_Gene_Drug.csv
DRKG/   DRKG_Gene_Biological_Process.csv  DRKG/   DRKG_Gene_Cellular_Component.csv
DRKG/   DRKG_Disease_Disease.csv       DRKG/   DRKG_Gene_Gene.csv
DRKG/   DRKG_Drug_Disease.csv          DRKG/   DRKG_Chemical_Chemical.csv
DRKG/   DRKG_Anatomy_Gene.csv          DRKG/   DRKG_Gene_Mol_Func.csv
DRKG/   DRKG_ChemicalEntity_Gene.csv   DRKG/   DRKG_Compound_SideEffect.csv
```


---
## 0 · Configuration — edit ONLY these two lines

In [34]:
import os
import re
import glob
import numpy as np
import pandas as pd

pd.set_option('future.no_silent_downcasting', True)  # suppress fillna FutureWarning

# ─────────────────────────────────────────────────────────────────────────────
# USER CONFIGURATION
# BASE_PATH : root folder containing all raw input data
# OUT_PATH  : folder where all processed CSVs will be saved
# ─────────────────────────────────────────────────────────────────────────────
BASE_PATH = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/"
OUT_PATH  = "/storage/Arushi/090526_EvoAge/kg_formation/processed_data/DRKG/"

# ── Derived input paths (do not edit below this line) ────────────────────────
DRKG_TSV_PATH      = os.path.join(BASE_PATH, "drkg/drkg.tsv")
DRKG_RELGLOS_PATH  = os.path.join(BASE_PATH, "drkg/relation_glossary.tsv")
DRKG_RELWISE_DIR   = os.path.join(BASE_PATH, "drkg/DRKG_relation_wise/")
PUBCHEM_PKL_PATH   = os.path.join(BASE_PATH, "databases_for_mapping/pubchem/combined_df.pkl")
PUBCHEM_DB_PATH    = os.path.join(BASE_PATH, "databases_for_mapping/pubchem/Pubchem_ID_DrugBank_Chebi.csv")
PUBCHEM_SYN_PATH   = os.path.join(BASE_PATH, "databases_for_mapping/pubchem/CID-Synonym-filtered")
ENS2NCBI_PATH      = os.path.join(BASE_PATH, "databases_for_mapping/ENSEMBL/ENSEMBLE_ID_2_NCBI_Gene_SYM.csv")
NCBI_HUMAN_PATH    = os.path.join(BASE_PATH, "databases_for_mapping/ncbi/Homo_sapiens.gene_info")
OMIM_DO_PATH       = os.path.join(BASE_PATH, "databases_for_mapping/DO/HumanDiseaseOntology-main/DOreports/OMIMinDO.tsv")
MESH2DOID_PATH     = os.path.join(BASE_PATH, "databases_for_mapping/DO/HumanDiseaseOntology-main/DOreports/MESHinDO.tsv")
MEDGEN_PATH        = os.path.join(BASE_PATH, "databases_for_mapping/MESH/MeSH/MedGenIDMappings.txt")
MEDGEN_HPO_PATH    = os.path.join(BASE_PATH, "databases_for_mapping/MESH/MeSH/MedGen_HPO_Mapping.txt")
MESH_COMB_PATH     = os.path.join(BASE_PATH, "databases_for_mapping/MESH/MESH_Combined.csv")
MESH_SUPP_PATH     = os.path.join(BASE_PATH, "databases_for_mapping/MESH/Mesh_Supplementary_Info.csv")
DO_PATH            = os.path.join(BASE_PATH, "databases_for_mapping/DO/All_DO_ref_files.csv")
GO_PATH            = os.path.join(BASE_PATH, "databases_for_mapping/MESH/MESH_GO_ID_NAME.csv")
UBERON_PATH        = os.path.join(BASE_PATH, "databases_for_mapping/uberon/Uberon_ID_Name_non_dup.csv")
HPO_PATH           = os.path.join(BASE_PATH, "databases_for_mapping/hpo/HPO.csv")

os.makedirs(OUT_PATH, exist_ok=True)
os.makedirs(DRKG_RELWISE_DIR, exist_ok=True)

DESIRED_COLS = [
    "Head", "Relation", "Tail", "Head_type", "Relation_type", "Tail_type", "Source",
    "KG_Source", "Head_ID", "Head_Name", "Head_Detail_name", "Head_detail_name", "Head_ENS",
    "Tail_name", "Tail_detail_name", "Tail_DOID_Name", "Tail_Detail_name",
    "Tail_ENS", "Head_ID_IS", "Tail_ID_IS", "Head_Orignal", "Tail_Orignal"
]

print("Paths configured.")
print(f"  Input  root : {BASE_PATH}")
print(f"  Output dir  : {OUT_PATH}")

Paths configured.
  Input  root : /storage/Arushi/090526_EvoAge/kg_formation/data_collection/
  Output dir  : /storage/Arushi/090526_EvoAge/kg_formation/processed_data/DRKG/


---
## Part 1 · Load and split DRKG by relation type

DRKG uses `EntityType::ID` format for head and tail nodes,
and `Source::RelationType::HeadType:TailType` format for relation IDs.

In [2]:
# Load the full DRKG TSV (no header — columns are head, relation, tail)
DRKG_raw = pd.read_csv(DRKG_TSV_PATH, sep='\t', header=None)
DRKG_raw.columns = [0, 1, 2]
print(f"DRKG total edges: {len(DRKG_raw):,}")
print("Sample rows:")
print(DRKG_raw.head(3).to_string())

DRKG total edges: 5,874,261
Sample rows:
            0                               1           2
0  Gene::2157  bioarx::HumGenHumGen:Gene:Gene  Gene::2157
1  Gene::2157  bioarx::HumGenHumGen:Gene:Gene  Gene::5264
2  Gene::2157  bioarx::HumGenHumGen:Gene:Gene  Gene::2158


In [3]:
# ── Parse the relation string: 'Source::RelationType::HeadType:TailType' ──────
# Extract the last two colon-separated segments as HeadType_TailType for grouping
DRKG_raw['Relation'] = DRKG_raw[1].apply(
    lambda x: ':'.join(x.split(':')[-2:]) if isinstance(x, str) else x
)
DRKG_raw['Relation'] = DRKG_raw['Relation'].str.replace(':', '_', regex=False)

# Rename raw columns; split EntityType::ID into separate columns
DRKG_raw.rename(columns={1: 'Relation_type'}, inplace=True)
DRKG_raw[['Head_type', 'Head']] = DRKG_raw[0].str.split('::', n=1, expand=True)
DRKG_raw[['Tail_type', 'Tail']] = DRKG_raw[2].str.split('::', n=1, expand=True)

print("Relation distribution (top 20):")
print(DRKG_raw['Relation'].value_counts().head(20))

Relation distribution (top 20):
Relation
Gene_Gene                       2350931
Compound_Compound               1385757
Anatomy_Gene                     726495
Gene_Biological Process          559504
Compound_Gene                    184504
Compound_Side Effect             138944
Gene_Molecular Function           97222
Gene_Disease                      95399
Gene_Pathway                      84372
Compound_Disease                  83895
Gene_Cellular Component           73566
Disease_Gene                      28438
Gene_Compound                     26290
Compound_Atc                      15750
Gene_Tax                          14663
Disease_Anatomy                    3602
Disease_Symptom                    3357
Pharmacologic Class_Compound       1029
Disease_Disease                     543
Name: count, dtype: int64


In [4]:

# ── Inspect entity namespaces ─────────────────────────────────────────────────
print("Head entity types:", set(DRKG_raw['Head_type']))
print("\nTail entity types:", set(DRKG_raw['Tail_type']))

# Show relation glossary if available
try:
    RG = pd.read_csv(DRKG_RELGLOS_PATH, sep='\t')
    print(f"\nRelation glossary: {len(RG)} entries")
    print(RG.head())
except FileNotFoundError:
    print("relation_glossary.tsv not found — skipping")

Head entity types: {'Disease', 'Pharmacologic Class', 'Gene', 'Anatomy', 'Compound'}

Tail entity types: {'Disease', 'Biological Process', 'Symptom', 'Side Effect', 'Tax', 'Gene', 'Cellular Component', 'Pathway', 'Atc', 'Anatomy', 'Molecular Function', 'Compound'}

Relation glossary: 107 entries
                                Relation-name Data-source  \
0             DGIDB::ACTIVATOR::Gene:Compound       DGIDB   
1               DGIDB::AGONIST::Gene:Compound       DGIDB   
2  DGIDB::ALLOSTERIC MODULATOR::Gene:Compound       DGIDB   
3            DGIDB::ANTAGONIST::Gene:Compound       DGIDB   
4              DGIDB::ANTIBODY::Gene:Compound       DGIDB   

  Connected entity-types       Interaction-type  \
0          Compound:Gene             activation   
1          Compound:Gene                agonism   
2          Compound:Gene  allosteric modulation   
3          Compound:Gene             antagonism   
4          Compound:Gene               antibody   

                             

In [5]:
# ── Save per-relation CSV files ───────────────────────────────────────────────
saved = 0
for relation, df_group in DRKG_raw.groupby('Relation'):
    fname = os.path.join(DRKG_RELWISE_DIR, f"DRKG_relation_wise_{relation}.csv")
    df_group.to_csv(fname, index=False)
    saved += 1

print(f"Saved {saved} relation-wise files to: {DRKG_RELWISE_DIR}")

# Export relation type stats
vc = DRKG_raw['Relation'].value_counts().reset_index()
vc.columns = ['Value', 'Count']
vc.to_csv(os.path.join(OUT_PATH, 'DRKG_relation_Type_STATS.csv'), index=False)
print(f"Relation stats saved.")
del DRKG_raw

Saved 19 relation-wise files to: /storage/Arushi/090526_EvoAge/kg_formation/data_collection/drkg/DRKG_relation_wise/
Relation stats saved.


---
## Part 2 · Load reference dictionaries

In [6]:
# ── 2a. PubChem CID ↔ SMILES and CID → IUPAC ─────────────────────────────────
Pubchem = pd.read_pickle(PUBCHEM_PKL_PATH)
Pubchem_Smile_CID_Dict  = dict(zip(Pubchem['PUBCHEM_SMILES'], Pubchem['PUBCHEM_COMPOUND_CID']))
Pubchem_CID_Smile_Dict  = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_SMILES']))
Pubchem_IUPAC_CID_Dict  = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_IUPAC_NAME']))
del Pubchem
print("PubChem CID dicts loaded")

PubChem CID dicts loaded


In [7]:
# ── 2b. DrugBank -> PubChem CID and ChEBI -> PubChem CID ─────────────────────
pubchem = pd.read_csv(PUBCHEM_DB_PATH)
pubchem_DB    = pubchem[~pubchem['DRUGBANK_ID'].isna()]
pubchem_CHEBI = pubchem[~pubchem['CHEBI_ID'].isna()]
DB2PC_dict    = dict(zip(pubchem_DB['DRUGBANK_ID'],  pubchem_DB['ID']))
Chebi2PC_dict = dict(zip(pubchem_CHEBI['CHEBI_ID'], pubchem_CHEBI['ID']))
del pubchem
print(f"DrugBank->PubChem: {len(DB2PC_dict):,} | ChEBI->PubChem: {len(Chebi2PC_dict):,}")

/tmp/ipykernel_2717970/3939769346.py:2: DtypeWarning: Columns (1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  pubchem = pd.read_csv(PUBCHEM_DB_PATH)


DrugBank->PubChem: 10,702 | ChEBI->PubChem: 177,680


In [8]:
# ── 2c. PubChem synonym -> CID ────────────────────────────────────────────────
Pubchem_Syn_fil = pd.read_csv(PUBCHEM_SYN_PATH, sep='\t', header=None)
Pubchem_Syn_fil_dict = dict(zip(Pubchem_Syn_fil[1], Pubchem_Syn_fil[0]))
del Pubchem_Syn_fil
print(f"PubChem synonyms: {len(Pubchem_Syn_fil_dict):,}")

PubChem synonyms: 102,877,691


In [9]:
# ── 2d. ENSEMBL <-> NCBI gene crosswalk ──────────────────────────────────────
ENS_2NCBI = pd.read_csv(ENS2NCBI_PATH)
ENS_2NCBI = ENS_2NCBI[~ENS_2NCBI['name'].isna()]
NCBI_2ENS__dict = dict(zip(ENS_2NCBI['name'], ENS_2NCBI['initial_alias']))
del ENS_2NCBI
print(f"Symbol -> Ensembl: {len(NCBI_2ENS__dict):,}")

Symbol -> Ensembl: 40,940


In [10]:
# ── 2e. NCBI Human gene_info ──────────────────────────────────────────────────
# DRKG gene nodes use raw NCBI GeneIDs (integers) as their identifiers
NCBI_ALL_GENE = pd.read_csv(NCBI_HUMAN_PATH, sep='\t')
NCBI_ALL_GENE['ENSEMBLE_ID']   = NCBI_ALL_GENE['Symbol'].map(NCBI_2ENS__dict)
NCBI_ALL_GENEname_dict         = dict(zip(NCBI_ALL_GENE['GeneID'],  NCBI_ALL_GENE['description']))
NCBI_ALL_GENEIDname_dict       = dict(zip(NCBI_ALL_GENE['GeneID'],  NCBI_ALL_GENE['Symbol']))
NCBI_ALL_GENE_SYM_DES          = dict(zip(NCBI_ALL_GENE['Symbol'],  NCBI_ALL_GENE['description']))
print(f"NCBI Human genes: {len(NCBI_ALL_GENEname_dict):,}")

NCBI Human genes: 193,505


In [11]:
# ── 2f. OMIM -> DOID ──────────────────────────────────────────────────────────
All_ref_omim = pd.read_csv(OMIM_DO_PATH, sep='\t')
All_ref_omim.rename(columns={'ID': 'id', 'LABEL': 'label'}, inplace=True)
All_ref_omim['xrefs'] = All_ref_omim['xrefs'].str.split(', ')
All_ref_omim = All_ref_omim.explode('xrefs')
All_ref_omim['xrefs'] = All_ref_omim['xrefs'].str.replace('MIM', 'OMIM')
OMIM_ID_2_DOID_dict = dict(zip(All_ref_omim['xrefs'], All_ref_omim['id']))
print(f"OMIM -> DOID: {len(OMIM_ID_2_DOID_dict):,}")

OMIM -> DOID: 6,094


In [12]:
# ── 2g. MeSH -> DOID crosswalk ────────────────────────────────────────────────
Mesh2DOID = pd.read_csv(MESH2DOID_PATH, sep='\t')
Mesh2DOID.rename(columns={'ID': 'id', 'LABEL': 'label'}, inplace=True)
Mesh2DOID['xrefs'] = Mesh2DOID['xrefs'].str.split(', ')
Mesh2DOID = Mesh2DOID.explode('xrefs')
Mesh2DOID_dict = dict(zip(Mesh2DOID['xrefs'], Mesh2DOID['id']))
print(f"MeSH -> DOID: {len(Mesh2DOID_dict):,}")

MeSH -> DOID: 3,687


In [13]:
# ── 2h. MedGen CUI -> preferred name ──────────────────────────────────────────
MedGen = pd.read_csv(MEDGEN_PATH, sep='\t')
MedGen = MedGen.drop_duplicates(subset=['source_id', 'source'], keep='first')
MedGen_MeshID_name_dict = dict(zip(MedGen['source_id'], MedGen['pref_name']))
print(f"MedGen MeSH ID -> name: {len(MedGen_MeshID_name_dict):,}")

MedGen MeSH ID -> name: 394,895


In [14]:
# ── 2i. MeSH combined and supplementary ──────────────────────────────────────
MESH = pd.read_csv(MESH_COMB_PATH)
MESH_dict      = dict(zip(MESH['ID'],   MESH['Name']))
MESH_name_dict = MESH_dict.copy()
MESH['Pubchem_ID'] = MESH['Name'].map(Pubchem_Syn_fil_dict)
MESH_pubchem = MESH[~MESH['Pubchem_ID'].isna()].copy()
MESH_pubchem['Pubchem_ID'] = MESH_pubchem['Pubchem_ID'].astype(str).str.replace(r'\.0$', '', regex=True)
MESH_pubchem_dict = dict(zip(MESH_pubchem['ID'], MESH_pubchem['Pubchem_ID']))

Mesh_supp = pd.read_csv(MESH_SUPP_PATH)
Mesh_supp['Pubchem_ID'] = Mesh_supp['Name'].map(Pubchem_Syn_fil_dict)
Mesh_supp_dict = dict(zip(Mesh_supp['ID'], Mesh_supp['Name']))
Mesh_pubchem_supp = Mesh_supp[~Mesh_supp['Pubchem_ID'].isna()].copy()
Mesh_pubchem_supp['Pubchem_ID'] = Mesh_pubchem_supp['Pubchem_ID'].astype(str).str.replace(r'\.0$', '', regex=True)
MESH_pubchem_Supp_dict = dict(zip(Mesh_pubchem_supp['ID'], Mesh_pubchem_supp['Pubchem_ID']))
print(f"MeSH combined: {len(MESH_dict):,} | MeSH->PubChem: {len(MESH_pubchem_dict):,}")

MeSH combined: 38,520 | MeSH->PubChem: 2,533


In [15]:
# ── 2j. Disease Ontology (DO) ─────────────────────────────────────────────────
DO_All_File = pd.read_csv(DO_PATH)
DOID_disname_dict      = dict(zip(DO_All_File['id'],    DO_All_File['label']))
DOID_disAltID_dict     = dict(zip(DO_All_File['id'],    DO_All_File['xrefs']))
DOID_disname_DOID_dict = dict(zip(DO_All_File['label'], DO_All_File['id']))
print(f"DO entries: {len(DOID_disname_DOID_dict):,}")

DO entries: 11,787


In [16]:
# ── 2k. GO terms ──────────────────────────────────────────────────────────────
GO = pd.read_csv(GO_PATH)
GO_dict = dict(zip(GO['id'], GO['name']))

# Secondary GO ID -> primary GO ID resolution
GO_SecID = GO[['id','name','namespace','alt_id']].dropna(subset=['alt_id']).copy()
GO_SecID['alt_id'] = GO_SecID['alt_id'].str.split(r'\s*\|\s*')
GO_SecID = GO_SecID.explode('alt_id')
GO_SecID_2_prim_ID_dict = dict(zip(GO_SecID['alt_id'], GO_SecID['id']))
print(f"GO terms: {len(GO_dict):,} | GO secondary IDs: {len(GO_SecID_2_prim_ID_dict):,}")

GO terms: 47,995 | GO secondary IDs: 3,646


In [17]:
# ── 2l. UBERON anatomy ────────────────────────────────────────────────────────
UBERON = pd.read_csv(UBERON_PATH)
UBERON_dict = dict(zip(UBERON['UBERON ID'], UBERON['Name']))
print(f"UBERON: {len(UBERON_dict):,}")

UBERON: 15,959


In [18]:
# ── 2m. HPO phenotype: UMLS -> HPO ID ────────────────────────────────────────
phenotype = pd.read_csv(HPO_PATH)
phenotype = phenotype[phenotype['id'].str.startswith('HP')]

def extract_umls(xref):
    """Extract UMLS CUI from xref string like 'UMLS:C0001234'."""
    if isinstance(xref, str):
        m = re.search(r'UMLS:[A-Za-z0-9]+', xref)
        if m:
            return m.group(0).replace('UMLS:', '')
    return None

phenotype['UMLS_ID'] = phenotype['xref'].apply(extract_umls)
phenotype_HPO_UMLS = phenotype[~phenotype['UMLS_ID'].isna()].copy()
UMLS_2_HPO_phenotype_dict = dict(zip(phenotype_HPO_UMLS['UMLS_ID'], phenotype_HPO_UMLS['id']))

# MedGen CUI -> HPO SDUI (fallback for Compound_Side_Effect)
MedGen_HPO = pd.read_csv(MEDGEN_HPO_PATH, sep='|')
MedGen_CUID_Source_ID_dict = dict(zip(MedGen_HPO['#CUI'], MedGen_HPO['SDUI']))
print(f"UMLS->HPO: {len(UMLS_2_HPO_phenotype_dict):,} | MedGen CUI->HPO: {len(MedGen_CUID_Source_ID_dict):,}")

UMLS->HPO: 11,411 | MedGen CUI->HPO: 18,795


---
## Part 3 · Helper functions

In [25]:
def gene_map(df, col):
    """
    Resolve a DRKG gene column (raw NCBI GeneID strings) to canonical Symbol.
    Steps:
      1. Remove rows containing ';' (multi-gene entries not resolvable to a single gene)
      2. Convert to integer (drops non-numeric IDs from non-human species)
      3. Map GeneID -> Symbol ({col}_Name)
      4. Map GeneID -> description ({col}_Detail_name)
      5. Map Symbol -> Ensembl ID ({col}_ENS)
    Only human genes appear in NCBI_ALL_GENEIDname_dict; non-human entries get NaN
    in {col}_Name and are filtered out by the caller.
    """
    df = df[~df[col].astype(str).str.contains(';')].copy()
    df[col] = df[col].astype(int)
    df[f'{col}_Name']        = df[col].map(NCBI_ALL_GENEIDname_dict)
    df[f'{col}_Detail_name'] = df[col].map(NCBI_ALL_GENEname_dict)
    df[f'{col}_ENS']         = df[f'{col}_Name'].map(NCBI_2ENS__dict)
    return df


def resolve_disease_col(df, col):
    """
    Resolve a disease column to DOID (preferred) or MeSH ID (fallback).
    Steps:
      1. Strip 'MESH:' prefix (DRKG uses both 'MESH:D001234' and bare 'D001234')
      2. Annotate with human-readable name (DOID label or MeSH name)
      3. Cast to str so .str.startswith() works reliably
      4. Set {col}_ID_IS = 'DOID' if starts with DOID, else 'MESH'
    Uses None not np.nan to avoid DTypePromotionError in numpy 2.0+.
    """
    df = df.copy()
    name_col  = f'{col}_detail_name'
    id_is_col = f'{col}_ID_IS'
    df[col] = df[col].astype(str).str.replace('MESH:', '', regex=False)
    df[name_col] = df[col].apply(
        lambda x: DOID_disname_dict.get(x) if isinstance(x, str) and x.startswith('DOID')
                  else (MESH_dict.get(x) or Mesh_supp_dict.get(x))
    )
    # FIX: cast to str so .str.startswith() works; use None not np.nan
    df[col] = df[col].astype(str)
    df[id_is_col] = np.where(
        df[col].str.startswith('DOID'), 'DOID', None)
    return df


def resolve_compound_head(df):
    """
    Resolve DRKG compound Head -> PubChem CID via cascade:
    DrugBank -> PubChem, ChEBI -> PubChem, SMILES -> PubChem (legacy).

    Replaced with Pubchem_Smile_CID_Dict (from loaded PubChem pickle).
    """
    df = df.copy()
    df['Head'] = df['Head'].str.replace(r'\s*\|.*$', '', regex=True)  # strip pipe-delimited aliases
    df['Head'] = df['Head'].str.replace('MESH:', '', regex=False)

    def map_compound(row):
        h = row['Head']
        if not isinstance(h, str):
            return np.nan
        if h.startswith('DB'):
            return DB2PC_dict.get(h, h)
        if h.startswith('CHEBI'):
            v = Chebi2PC_dict.get(h, np.nan)
            return str(v).rstrip('.0') if v is not np.nan else np.nan
        if h.startswith('C') or h.startswith('D'):
            pc = MESH_pubchem_dict.get(h) or MESH_pubchem_Supp_dict.get(h)
            return pc if pc else np.nan
        # Fallback: SMILES -> CID (for legacy DRKG compound IDs)
        smiles = row.get('Head_smiles', np.nan)
        return Pubchem_Smile_CID_Dict.get(smiles, np.nan) if pd.notna(smiles) else np.nan

    df['Head_ID'] = df.apply(map_compound, axis=1)
    df['Head_ID'] = df['Head_ID'].astype(str).str.rstrip('.0').replace('nan', np.nan)
    df = df[~df['Head_ID'].isna()]
    df[['Head', 'Head_ID']] = df[['Head_ID', 'Head']]
    df['Head'] = df['Head'].astype(str)
    df['Head_ID_IS'] = np.where(df['Head'].str.startswith('DB'), 'Drugbank', 'Pubchem')
    return df


def select_cols(df):
    return df[[c for c in DESIRED_COLS if c in df.columns]]


def load_rel(relation_name):
    """Load a per-relation CSV from DRKG_relation_wise folder."""
    path = os.path.join(DRKG_RELWISE_DIR, f"DRKG_relation_wise_{relation_name}.csv")
    df = pd.read_csv(path)
    return df


def save(df, filename):
    path = os.path.join(OUT_PATH, filename)
    df.to_csv(path, index=None)
    print(f"Saved {len(df):,} rows -> {path}")


print("Helper functions defined.")

Helper functions defined.


---
## Part 4 · Load per-relation DataFrames

In [27]:
import glob as glob_mod

variable_names = []
file_list = glob_mod.glob(os.path.join(DRKG_RELWISE_DIR, "DRKG_relation_wise_*.csv"))

for file_path in file_list:
    base_name = os.path.basename(file_path).replace(".csv", "")
    var_name  = base_name.replace("-", "_").replace("DRKG_relation_wise_", "df_")
    globals()[var_name] = pd.read_csv(file_path)
    variable_names.append(var_name)

print(f"Loaded {len(variable_names)} relation DataFrames:")
for name in sorted(variable_names):
    print(f"  {name}")

/tmp/ipykernel_2717970/3903473135.py:9: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  globals()[var_name] = pd.read_csv(file_path)
/tmp/ipykernel_2717970/3903473135.py:9: DtypeWarning: Columns (5,7) have mixed types. Specify dtype option on import or set low_memory=False.
  globals()[var_name] = pd.read_csv(file_path)


Loaded 23 relation DataFrames:
  df_Anatomy_Gene
  df_Compound_Atc
  df_Compound_Compound
  df_Compound_Disease
  df_Compound_Gene
  df_Compound_Side Effect
  df_Compound_SideEffect
  df_Disease_Anatomy
  df_Disease_Disease
  df_Disease_Gene
  df_Disease_Symptom
  df_Gene_Biological Process
  df_Gene_BiologicalProcess
  df_Gene_Cellular Component
  df_Gene_CellularComponent
  df_Gene_Compound
  df_Gene_Disease
  df_Gene_Gene
  df_Gene_Molecular Function
  df_Gene_MolecularFunction
  df_Gene_Pathway
  df_Gene_Tax
  df_Pharmacologic Class_Compound


---
## Part 5 · Process and export each relation type

### 1 · Gene_Disease

In [28]:
df_Gene_Disease = gene_map(df_Gene_Disease, 'Head')
df_Gene_Disease = df_Gene_Disease[~df_Gene_Disease['Head_Name'].isna()]  # keep Human genes only

df_Gene_Disease = resolve_disease_col(df_Gene_Disease, 'Tail')
df_Gene_Disease = df_Gene_Disease[~df_Gene_Disease['Tail_detail_name'].isna()]

# Resolve disease name -> DOID where possible
df_Gene_Disease['Tail_DOID_Name'] = (df_Gene_Disease['Tail_detail_name']
    .map(DOID_disname_DOID_dict).fillna(df_Gene_Disease['Tail']))

df_Gene_Disease[['Head', 'Head_Name']] = df_Gene_Disease[['Head_Name', 'Head']]
df_Gene_Disease[['Tail', 'Tail_DOID_Name']] = df_Gene_Disease[['Tail_DOID_Name', 'Tail']]
df_Gene_Disease.rename(columns={'0':'Head_Orignal','2':'Tail_Orignal','Head_Name':'Head_ID'}, inplace=True, errors='ignore')
df_Gene_Disease['KG_Source']  = 'DRKG'
df_Gene_Disease['Head_ID_IS'] = 'NCBI_ID'

df_Gene_Disease = select_cols(df_Gene_Disease)
df_Gene_Disease.rename(columns={'Tail_detail_name':'Tail_detail_name'}, inplace=True, errors='ignore')
save(df_Gene_Disease, 'DRKG_Gene_Disease.csv')

Saved 63,439 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/DRKG/DRKG_Gene_Disease.csv


### 2 · Disease_Anatomy

In [ ]:
Head_Detail_name", "Head_ENS",
    "Tail_name", "Tail_detail_name

In [29]:
df_Disease_Anatomy['Tail_detail_name'] = df_Disease_Anatomy['Tail'].map(UBERON_dict)
df_Disease_Anatomy = resolve_disease_col(df_Disease_Anatomy, 'Head')
df_Disease_Anatomy = df_Disease_Anatomy[~df_Disease_Anatomy['Head_detail_name'].isna()]
df_Disease_Anatomy.rename(columns={'0':'Head_Orignal','2':'Tail_Orignal'}, inplace=True, errors='ignore')
df_Disease_Anatomy['KG_Source']  = 'DRKG'
df_Disease_Anatomy['Tail_ID_IS'] = 'UBERON_ID'
df_Disease_Anatomy
df_Disease_Anatomy.rename(columns={'Head_detail_name':'Head_Detail_name','Tail_detail_name':'Tail_detail_name'}, inplace=True, errors='ignore')
df_Disease_Anatomy = select_cols(df_Disease_Anatomy)
save(df_Disease_Anatomy, 'DRKG_Disease_Anatomy.csv')

Saved 3,591 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/DRKG/DRKG_Disease_Anatomy.csv


### 3 · Gene_Biological_Process

In [48]:
df_Gene_Biological_Process = gene_map(df_Gene_BiologicalProcess, 'Head')
df_Gene_Biological_Process = df_Gene_Biological_Process[~df_Gene_Biological_Process['Head_Name'].isna()]

# Resolve GO secondary IDs to primary IDs where needed
df_Gene_Biological_Process['Tail'] = df_Gene_Biological_Process['Tail'].map(
    GO_SecID_2_prim_ID_dict).fillna(df_Gene_Biological_Process['Tail'])
df_Gene_Biological_Process['Tail_Detail_name'] = df_Gene_Biological_Process['Tail'].map(GO_dict)
df_Gene_Biological_Process = df_Gene_Biological_Process[~df_Gene_Biological_Process['Tail_Detail_name'].isna()]

df_Gene_Biological_Process[['Head', 'Head_Name']] = df_Gene_Biological_Process[['Head_Name', 'Head']]
df_Gene_Biological_Process.rename(columns={'0':'Head_Orignal','2':'Tail_Orignal'}, inplace=True, errors='ignore')
df_Gene_Biological_Process['KG_Source']  = 'DRKG'
df_Gene_Biological_Process['Head_ID_IS'] = 'NCBI_ID'
df_Gene_Biological_Process['Tail_ID_IS'] = 'Quick_GO'
df_Gene_Biological_Process = select_cols(df_Gene_Biological_Process)
save(df_Gene_Biological_Process, 'DRKG_Gene_Biological_Process.csv')

Saved 559,453 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/DRKG/DRKG_Gene_Biological_Process.csv


### 4 · Disease_Disease

In [32]:
df_Disease_Disease
df_Disease_Disease = resolve_disease_col(df_Disease_Disease, 'Head')
df_Disease_Disease = resolve_disease_col(df_Disease_Disease, 'Tail')

In [35]:

df_Disease_Disease = df_Disease_Disease[~df_Disease_Disease['Head_detail_name'].isna()]
df_Disease_Disease = df_Disease_Disease[~df_Disease_Disease['Tail_detail_name'].isna()]
df_Disease_Disease.rename(columns={'0':'Head_Orignal','2':'Tail_Orignal'}, inplace=True, errors='ignore')
df_Disease_Disease['KG_Source'] = 'DRKG'
df_Disease_Disease = select_cols(df_Disease_Disease)
df_Disease_Disease

Saved 535 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/DRKG/DRKG_Disease_Disease.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,KG_Source,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS,Head_Orignal,Tail_Orignal
0,D009373,Disease_Disease,DOID:11934,Disease,Hetionet::DrD::Disease:Disease,Disease,DRKG,"Neoplasms, Germ Cell and Embryonal",head and neck cancer,None,DOID,Disease::MESH:D009373,Disease::DOID:11934
1,D015179,Disease_Disease,DOID:3571,Disease,Hetionet::DrD::Disease:Disease,Disease,DRKG,Colorectal Neoplasms,liver cancer,None,DOID,Disease::MESH:D015179,Disease::DOID:3571
2,D007680,Disease_Disease,DOID:4045,Disease,Hetionet::DrD::Disease:Disease,Disease,DRKG,Kidney Neoplasms,muscle cancer,None,DOID,Disease::MESH:D007680,Disease::DOID:4045
3,D005185,Disease_Disease,DOID:13223,Disease,Hetionet::DrD::Disease:Disease,Disease,DRKG,Fallopian Tube Neoplasms,uterine fibroid,None,DOID,Disease::MESH:D005185,Disease::DOID:13223
4,D001859,Disease_Disease,DOID:10283,Disease,Hetionet::DrD::Disease:Disease,Disease,DRKG,Bone Neoplasms,prostate cancer,None,DOID,Disease::MESH:D001859,Disease::DOID:10283
...,...,...,...,...,...,...,...,...,...,...,...,...,...
538,D015179,Disease_Disease,DOID:8577,Disease,Hetionet::DrD::Disease:Disease,Disease,DRKG,Colorectal Neoplasms,ulcerative colitis,None,DOID,Disease::MESH:D015179,Disease::DOID:8577
539,D009373,Disease_Disease,DOID:13499,Disease,Hetionet::DrD::Disease:Disease,Disease,DRKG,"Neoplasms, Germ Cell and Embryonal",jejunal cancer,None,DOID,Disease::MESH:D009373,Disease::DOID:13499
540,D010190,Disease_Disease,DOID:10534,Disease,Hetionet::DrD::Disease:Disease,Disease,DRKG,Pancreatic Neoplasms,stomach cancer,None,DOID,Disease::MESH:D010190,Disease::DOID:10534
541,D015179,Disease_Disease,DOID:3121,Disease,Hetionet::DrD::Disease:Disease,Disease,DRKG,Colorectal Neoplasms,gallbladder cancer,None,DOID,Disease::MESH:D015179,Disease::DOID:3121


In [38]:
df_Disease_Gene['Head_ID_IS'] = np.where(df_Disease_Gene['Head'].str.startswith('DOID'), 'DO_ID', 'MESH')
df_Disease_Gene = select_cols(df_Disease_Gene)
df_Disease_Gene

,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Head_ID_IS
0,SARS-CoV2 E,Disease_Gene,8546,Disease,bioarx::Covid2_acc_host_gene::Disease:Gene,Gene,MESH
1,SARS-CoV2 E,Disease_Gene,23476,Disease,bioarx::Covid2_acc_host_gene::Disease:Gene,Gene,MESH
2,SARS-CoV2 E,Disease_Gene,6046,Disease,bioarx::Covid2_acc_host_gene::Disease:Gene,Gene,MESH
3,SARS-CoV2 E,Disease_Gene,10283,Disease,bioarx::Covid2_acc_host_gene::Disease:Gene,Gene,MESH
4,SARS-CoV2 E,Disease_Gene,124245,Disease,bioarx::Covid2_acc_host_gene::Disease:Gene,Gene,MESH
...,...,...,...,...,...,...,...
28433,MESH:D054990,Disease_Gene,66004,Disease,Hetionet::DuG::Disease:Gene,Gene,MESH
28434,MESH:D008175,Disease_Gene,60625,Disease,Hetionet::DuG::Disease:Gene,Gene,MESH
28435,MESH:D053713,Disease_Gene,90701,Disease,Hetionet::DuG::Disease:Gene,Gene,MESH
28436,MESH:D015535,Disease_Gene,158431,Disease,Hetionet::DuG::Disease:Gene,Gene,MESH


In [39]:
save(df_Disease_Disease, 'DRKG_Disease_Disease.csv')


Saved 535 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/DRKG/DRKG_Disease_Disease.csv


### 5 · Compound_Disease → Drug_Disease

In [32]:
df_Compound_Disease = resolve_compound_head(df_Compound_Disease)

# Resolve disease Tail: OMIM -> DOID, MESH -> DOID, keep DOID as-is
# Drop OMIM rows (not resolvable via MESH crosswalk)
df_Compound_Disease = df_Compound_Disease[~df_Compound_Disease['Tail'].str.startswith('OMIM')]
df_Compound_Disease['Tail'] = df_Compound_Disease['Tail'].str.replace('MESH:', '', regex=False)
df_Compound_Disease['Tail_DOID'] = df_Compound_Disease['Tail'].map(Mesh2DOID_dict).fillna(df_Compound_Disease['Tail'])
df_Compound_Disease['Tail_detail_name'] = df_Compound_Disease['Tail'].apply(
    lambda x: DOID_disname_dict.get(x) if isinstance(x, str) and x.startswith('DOID')
              else (MESH_dict.get(x) or Mesh_supp_dict.get(x))
)
df_Compound_Disease.rename(columns={'0':'Head_Orignal','2':'Tail_Orignal'}, inplace=True, errors='ignore')
df_Compound_Disease['KG_Source'] = 'DRKG'
# FIX: use None not np.nan
df_Compound_Disease['Tail'] = df_Compound_Disease['Tail'].astype(str)
df_Compound_Disease['Tail_ID_IS'] = np.where(
    df_Compound_Disease['Tail'].str.startswith('DOID'), 'DO_ID', 'MESH')
df_Compound_Disease = select_cols(df_Compound_Disease)
save(df_Compound_Disease, 'DRKG_Drug_Disease.csv')

Saved 58,032 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/DRKG/DRKG_Drug_Disease.csv


### 6 · Disease_Gene

In [33]:
df_Disease_Gene = gene_map(df_Disease_Gene, 'Tail')

df_Disease_Gene['Head'] = df_Disease_Gene['Head'].map(Mesh2DOID_dict).fillna(df_Disease_Gene['Head'])
df_Disease_Gene['Head'] = df_Disease_Gene['Head'].str.replace('MESH:', '', regex=False)
df_Disease_Gene['Head_Detail_name'] = df_Disease_Gene['Head'].apply(
    lambda x: DOID_disname_dict.get(x) if isinstance(x, str) and x.startswith('DOID')
              else (MESH_dict.get(x) or Mesh_supp_dict.get(x))
)
# Keep SARS entries that have no DO/MeSH mapping as-is
df_Disease_Gene.loc[
    df_Disease_Gene['Head'].str.startswith('SARS') & df_Disease_Gene['Head_Detail_name'].isna(),
    'Head_Detail_name'] = df_Disease_Gene['Head']
df_Disease_Gene = df_Disease_Gene[~df_Disease_Gene['Head_Detail_name'].isna()]

# Resolve gene Tail: GeneID -> Symbol -> description
df_Disease_Gene['Tail_ID'] = df_Disease_Gene['Tail'].map(NCBI_ALL_GENEIDname_dict)
df_Disease_Gene = df_Disease_Gene[~df_Disease_Gene['Tail_ID'].isna()]
df_Disease_Gene[['Tail', 'Tail_ID']] = df_Disease_Gene[['Tail_ID', 'Tail']]
df_Disease_Gene['Tail_Detail_name'] = df_Disease_Gene['Tail'].map(NCBI_ALL_GENE_SYM_DES)
df_Disease_Gene = df_Disease_Gene[~df_Disease_Gene['Tail_Detail_name'].isna()]

df_Disease_Gene.rename(columns={'0':'Head_Orignal','2':'Tail_Orignal'}, inplace=True, errors='ignore')
df_Disease_Gene['KG_Source']  = 'DRKG'
df_Disease_Gene['Tail_ID_IS'] = 'NCBI_ID'
df_Disease_Gene['Head'] = df_Disease_Gene['Head'].astype(str)
df_Disease_Gene['Head_ID_IS'] = np.where(df_Disease_Gene['Head'].str.startswith('DOID'), 'DO_ID', 'MESH')
df_Disease_Gene = select_cols(df_Disease_Gene)
save(df_Disease_Gene, 'DRKG_Disease_Gene.csv')

Saved 28,388 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/DRKG/DRKG_Disease_Gene.csv


### 7 · Gene_Compound → Gene_Drug

In [34]:
df_Gene_Compound = gene_map(df_Gene_Compound, 'Head')
df_Gene_Compound = df_Gene_Compound[~df_Gene_Compound['Head_Name'].isna()]

# Replaced with Pubchem_Smile_CID_Dict from the loaded PubChem pickle.
df_Gene_Compound['Tail_smiles'] = df_Gene_Compound['Tail'].map(Pubchem_Smile_CID_Dict)

def map_compound_tail(row):
    t = row['Tail']
    if not isinstance(t, str): return np.nan
    if t.startswith('DB'):
        return DB2PC_dict.get(t, t)
    if t.startswith('CHEBI'):
        v = Chebi2PC_dict.get(t, np.nan)
        return str(v).rstrip('.0') if v is not np.nan else np.nan
    smiles = row.get('Tail_smiles', np.nan)
    return Pubchem_Smile_CID_Dict.get(smiles, np.nan) if pd.notna(smiles) else np.nan

df_Gene_Compound['Tail_ID'] = df_Gene_Compound.apply(map_compound_tail, axis=1)
df_Gene_Compound['Tail_ID'] = df_Gene_Compound['Tail_ID'].replace('nan', np.nan)
df_Gene_Compound = df_Gene_Compound[~df_Gene_Compound['Tail_ID'].isna()]

df_Gene_Compound[['Head', 'Head_Name']] = df_Gene_Compound[['Head_Name', 'Head']]
df_Gene_Compound[['Tail', 'Tail_ID']] = df_Gene_Compound[['Tail_ID', 'Tail']]
df_Gene_Compound['Tail_Detail_name'] = df_Gene_Compound['Tail'].astype(str).map(Pubchem_IUPAC_CID_Dict)
df_Gene_Compound.rename(columns={'0':'Head_Orignal','2':'Tail_Orignal'}, inplace=True, errors='ignore')
df_Gene_Compound['KG_Source']  = 'DRKG'
df_Gene_Compound['Head_ID_IS'] = 'NCBI_ID'
df_Gene_Compound['Tail'] = df_Gene_Compound['Tail'].astype(str)
df_Gene_Compound['Tail_ID_IS'] = np.where(df_Gene_Compound['Tail'].str.startswith('DB'), 'DRUGBANK', 'Pubchem_ID')
df_Gene_Compound = select_cols(df_Gene_Compound)
save(df_Gene_Compound, 'DRKG_Gene_Drug.csv')

Saved 16,621 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/DRKG/DRKG_Gene_Drug.csv


### 8 · Gene_Cellular_Component

In [49]:
df_Gene_Cellular_Component = gene_map(df_Gene_CellularComponent, 'Head')
df_Gene_Cellular_Component['Tail_Detail_name'] = df_Gene_Cellular_Component['Tail'].map(GO_dict)
df_Gene_Cellular_Component = df_Gene_Cellular_Component[~df_Gene_Cellular_Component['Head_Name'].isna()]
df_Gene_Cellular_Component = df_Gene_Cellular_Component[~df_Gene_Cellular_Component['Tail_Detail_name'].isna()]
df_Gene_Cellular_Component[['Head', 'Head_Name']] = df_Gene_Cellular_Component[['Head_Name', 'Head']]
df_Gene_Cellular_Component.rename(columns={'0':'Head_Orignal','2':'Tail_Orignal'}, inplace=True, errors='ignore')
df_Gene_Cellular_Component['KG_Source']  = 'DRKG'
df_Gene_Cellular_Component['Head_ID_IS'] = 'NCBI_ID'
df_Gene_Cellular_Component['Tail_ID_IS'] = 'Quick_GO'
df_Gene_Cellular_Component = select_cols(df_Gene_Cellular_Component)
save(df_Gene_Cellular_Component, 'DRKG_Gene_Cellular_Component.csv')

Saved 71,088 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/DRKG/DRKG_Gene_Cellular_Component.csv


### 9 · Gene_Gene

In [39]:
# Keep only numeric Gene IDs in both columns
df_Gene_Gene = df_Gene_Gene[pd.to_numeric(df_Gene_Gene['Head'], errors='coerce').notna()]
df_Gene_Gene = df_Gene_Gene[pd.to_numeric(df_Gene_Gene['Tail'], errors='coerce').notna()]
df_Gene_Gene['Head'] = df_Gene_Gene['Head'].astype(str).str.replace(r'\.0$', '', regex=True)
df_Gene_Gene['Tail'] = df_Gene_Gene['Tail'].astype(str).str.replace(r'\.0$', '', regex=True)

df_Gene_Gene = gene_map(df_Gene_Gene, 'Head')
df_Gene_Gene = gene_map(df_Gene_Gene, 'Tail')

df_Gene_Gene[['Head', 'Head_Name']] = df_Gene_Gene[['Head_Name', 'Head']]
df_Gene_Gene[['Tail', 'Tail_Name']] = df_Gene_Gene[['Tail_Name', 'Tail']]
df_Gene_Gene = df_Gene_Gene[~df_Gene_Gene['Head'].isna()]
df_Gene_Gene = df_Gene_Gene[~df_Gene_Gene['Tail'].isna()]

df_Gene_Gene.rename(columns={'0':'Head_Orignal','2':'Tail_Orignal'}, inplace=True, errors='ignore')
df_Gene_Gene['KG_Source']  = 'DRKG'
df_Gene_Gene['Head_ID_IS'] = 'NCBI_ID'
df_Gene_Gene['Tail_ID_IS'] = 'NCBI_ID'
df_Gene_Gene = select_cols(df_Gene_Gene)
save(df_Gene_Gene, 'DRKG_Gene_Gene.csv')

Saved 2,326,174 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/DRKG/DRKG_Gene_Gene.csv


### 10 · Compound_Compound → Chemical_Chemical

In [40]:
def map_db(x):
    if isinstance(x, str) and x.startswith('DB'):
        return DB2PC_dict.get(x, x)
    return np.nan

df_Compound_Compound['Head_ID'] = df_Compound_Compound['Head'].apply(map_db)
df_Compound_Compound['Tail_ID'] = df_Compound_Compound['Tail'].apply(map_db)
df_Compound_Compound[['Head', 'Head_ID']] = df_Compound_Compound[['Head_ID', 'Head']]
df_Compound_Compound[['Tail', 'Tail_ID']] = df_Compound_Compound[['Tail_ID', 'Tail']]
df_Compound_Compound.rename(columns={'0':'Head_Orignal','2':'Tail_Orignal'}, inplace=True, errors='ignore')
df_Compound_Compound['KG_Source'] = 'DRKG'
df_Compound_Compound['Head'] = df_Compound_Compound['Head'].astype(str)
df_Compound_Compound['Tail'] = df_Compound_Compound['Tail'].astype(str)
df_Compound_Compound['Head_ID_IS'] = df_Compound_Compound['Head'].apply(
    lambda x: 'Drugbank' if x.startswith('DB') else 'Pubchem')
df_Compound_Compound['Tail_ID_IS'] = np.where(
    df_Compound_Compound['Tail'].str.startswith('DB'), 'Drugbank', 'Pubchem')
df_Compound_Compound = select_cols(df_Compound_Compound)
save(df_Compound_Compound, 'DRKG_Chemical_Chemical.csv')

Saved 1,385,757 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/DRKG/DRKG_Chemical_Chemical.csv


### 11 · Anatomy_Gene

In [41]:
df_Anatomy_Gene = gene_map(df_Anatomy_Gene, 'Tail')
df_Anatomy_Gene['Head_detail_name'] = df_Anatomy_Gene['Head'].map(UBERON_dict)
df_Anatomy_Gene = df_Anatomy_Gene[~df_Anatomy_Gene['Head_detail_name'].isna()]
df_Anatomy_Gene = df_Anatomy_Gene[~df_Anatomy_Gene['Tail_Detail_name'].isna()]
df_Anatomy_Gene[['Tail', 'Tail_Name']] = df_Anatomy_Gene[['Tail_Name', 'Tail']]
df_Anatomy_Gene.rename(columns={'0':'Head_Orignal','2':'Tail_Orignal'}, inplace=True, errors='ignore')
df_Anatomy_Gene['KG_Source']  = 'DRKG'
df_Anatomy_Gene['Head_ID_IS'] = 'UBERON_ID'
df_Anatomy_Gene['Tail_ID_IS'] = 'NCBI_ID'
df_Anatomy_Gene = select_cols(df_Anatomy_Gene)
save(df_Anatomy_Gene, 'DRKG_Anatomy_Gene.csv')

Saved 726,156 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/DRKG/DRKG_Anatomy_Gene.csv


### 12 · Gene_Molecular_Function

In [50]:
df_Gene_Molecular_Function = gene_map(df_Gene_MolecularFunction, 'Head')
df_Gene_Molecular_Function['Tail_Detail_name'] = df_Gene_Molecular_Function['Tail'].map(GO_dict)
df_Gene_Molecular_Function = df_Gene_Molecular_Function[~df_Gene_Molecular_Function['Head_Name'].isna()]
df_Gene_Molecular_Function = df_Gene_Molecular_Function[~df_Gene_Molecular_Function['Tail_Detail_name'].isna()]
df_Gene_Molecular_Function[['Head', 'Head_Name']] = df_Gene_Molecular_Function[['Head_Name', 'Head']]
df_Gene_Molecular_Function.rename(columns={'0':'Head_Orignal','2':'Tail_Orignal'}, inplace=True, errors='ignore')
df_Gene_Molecular_Function['KG_Source']  = 'DRKG'
df_Gene_Molecular_Function['Head_ID_IS'] = 'NCBI_ID'
df_Gene_Molecular_Function['Tail_ID_IS'] = 'Quick_GO'
df_Gene_Molecular_Function = select_cols(df_Gene_Molecular_Function)
save(df_Gene_Molecular_Function, 'DRKG_Gene_Mol_Func.csv')

Saved 86,685 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/DRKG/DRKG_Gene_Mol_Func.csv


### 13 · Compound_Gene → ChemicalEntity_Gene

In [29]:
# Keep only numeric Tail values (NCBI GeneIDs)
df_Compound_Gene['Tail'] = df_Compound_Gene['Tail'].astype(str).str.replace(r'\.0$', '', regex=True)
df_Compound_Gene = df_Compound_Gene[pd.to_numeric(df_Compound_Gene['Tail'], errors='coerce').notna()]
df_Compound_Gene['Tail'] = df_Compound_Gene['Tail'].astype(int)

df_Compound_Gene['Head_smiles'] = df_Compound_Gene['Head'].map(Pubchem_Smile_CID_Dict)

def map_compound_head(row):
    h = row['Head']
    if not isinstance(h, str): return np.nan
    if h.startswith('DB'):
        return DB2PC_dict.get(h, h)
    if h.startswith('CHEBI'):
        v = Chebi2PC_dict.get(h, np.nan)
        return str(v).rstrip('.0') if v is not np.nan else np.nan
    smiles = row.get('Head_smiles', np.nan)
    return Pubchem_Smile_CID_Dict.get(smiles, np.nan) if pd.notna(smiles) else np.nan

/tmp/ipykernel_2717970/4201695617.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_Compound_Gene['Tail'] = df_Compound_Gene['Tail'].astype(int)
/tmp/ipykernel_2717970/4201695617.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_Compound_Gene['Head_smiles'] = df_Compound_Gene['Head'].map(Pubchem_Smile_CID_Dict)


In [30]:
df_Compound_Gene

,0,Relation_type,2,Relation,Head_type,Head,Tail_type,Tail,Head_smiles
1165,Compound::DB02573,bioarx::DrugHumGen:Compound:Gene,Gene::6035,Compound_Gene,Compound,DB02573,Gene,6035,NaN
1166,Compound::molport:MolPort-003-808-703,bioarx::DrugHumGen:Compound:Gene,Gene::1814,Compound_Gene,Compound,molport:MolPort-003-808-703,Gene,1814,NaN
1167,Compound::molport:MolPort-003-808-703,bioarx::DrugHumGen:Compound:Gene,Gene::1813,Compound_Gene,Compound,molport:MolPort-003-808-703,Gene,1813,NaN
1168,Compound::molport:MolPort-003-808-703,bioarx::DrugHumGen:Compound:Gene,Gene::3269,Compound_Gene,Compound,molport:MolPort-003-808-703,Gene,3269,NaN
1169,Compound::DB13217,bioarx::DrugHumGen:Compound:Gene,Gene::11251,Compound_Gene,Compound,DB13217,Gene,11251,NaN
...,...,...,...,...,...,...,...,...,...
184499,Compound::DB00619,INTACT::PHYSICAL ASSOCIATION::Compound:Gene,Gene::780,Compound_Gene,Compound,DB00619,Gene,780,NaN
184500,Compound::DB00619,INTACT::PHYSICAL ASSOCIATION::Compound:Gene,Gene::84959,Compound_Gene,Compound,DB00619,Gene,84959,NaN
184501,Compound::CHEMBL9506,INTACT::PHYSICAL ASSOCIATION::Compound:Gene,Gene::886,Compound_Gene,Compound,CHEMBL9506,Gene,886,NaN
184502,Compound::DB01037,INTACT::PHYSICAL ASSOCIATION::Compound:Gene,Gene::4129,Compound_Gene,Compound,DB01037,Gene,4129,NaN


In [24]:
df_Compound_Gene['Head_ID'] = df_Compound_Gene.apply(map_compound_head, axis=1)
df_Compound_Gene['Head_ID'] = df_Compound_Gene['Head_ID'].replace('nan', np.nan)

df_Compound_Gene = gene_map(df_Compound_Gene, 'Tail')
df_Compound_Gene = df_Compound_Gene[~df_Compound_Gene['Head_ID'].isna()]
df_Compound_Gene = df_Compound_Gene[~df_Compound_Gene['Tail_Name'].isna()]

df_Compound_Gene[['Head', 'Head_ID']] = df_Compound_Gene[['Head_ID', 'Head']]
df_Compound_Gene[['Tail', 'Tail_Name']] = df_Compound_Gene[['Tail_Name', 'Tail']]
df_Compound_Gene['Tail_Detail_name'] = df_Compound_Gene['Tail'].map(NCBI_ALL_GENE_SYM_DES)

df_Compound_Gene.rename(columns={'0':'Head_Orignal','2':'Tail_Orignal'}, inplace=True, errors='ignore')
df_Compound_Gene['KG_Source']  = 'DRKG'
df_Compound_Gene['Tail_ID_IS'] = 'NCBI_ID'
df_Compound_Gene['Head'] = df_Compound_Gene['Head'].astype(str)

df_Compound_Gene['Head_ID_IS'] = df_Compound_Gene['Head'].apply(
    lambda x: 'Drugbank' if x.startswith('DB') else 'Pubchem')
df_Compound_Gene = select_cols(df_Compound_Gene)
df_Compound_Gene

,Head,Relation,Tail,Head_type,Relation_type,Tail_type,KG_Source,Head_ID,Tail_Detail_name,Tail_ENS,Head_ID_IS,Tail_ID_IS,Head_Orignal,Tail_Orignal
1165,448286,Compound_Gene,RNASE1,Compound,bioarx::DrugHumGen:Compound:Gene,Gene,DRKG,DB02573,"ribonuclease A family member 1, pancreatic",ENSG00000129538,Pubchem,NCBI_ID,Compound::DB02573,Gene::6035
1169,28871,Compound_Gene,PTGDR2,Compound,bioarx::DrugHumGen:Compound:Gene,Gene,DRKG,DB13217,prostaglandin D2 receptor 2,ENSG00000183134,Pubchem,NCBI_ID,Compound::DB13217,Gene::11251
1170,1948,Compound_Gene,MMP3,Compound,bioarx::DrugHumGen:Compound:Gene,Gene,DRKG,DB07986,matrix metallopeptidase 3,ENSG00000149968,Pubchem,NCBI_ID,Compound::DB07986,Gene::4314
1171,23586154,Compound_Gene,AFDN,Compound,bioarx::DrugHumGen:Compound:Gene,Gene,DRKG,DB08574,"afadin, adherens junction formation factor",ENSG00000130396,Pubchem,NCBI_ID,Compound::DB08574,Gene::4301
1191,6857692,Compound_Gene,F11,Compound,bioarx::DrugHumGen:Compound:Gene,Gene,DRKG,DB07077,coagulation factor XI,ENSG00000088926,Pubchem,NCBI_ID,Compound::DB07077,Gene::2160
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
184498,5291,Compound_Gene,SRC,Compound,INTACT::PHYSICAL ASSOCIATION::Compound:Gene,Gene,DRKG,DB00619,"SRC proto-oncogene, non-receptor tyrosine kinase",ENSG00000197122,Pubchem,NCBI_ID,Compound::DB00619,Gene::6714
184499,5291,Compound_Gene,DDR1,Compound,INTACT::PHYSICAL ASSOCIATION::Compound:Gene,Gene,DRKG,DB00619,discoidin domain receptor tyrosine kinase 1,ENSG00000204580,Pubchem,NCBI_ID,Compound::DB00619,Gene::780
184500,5291,Compound_Gene,UBASH3B,Compound,INTACT::PHYSICAL ASSOCIATION::Compound:Gene,Gene,DRKG,DB00619,ubiquitin associated and SH3 domain containing B,ENSG00000154127,Pubchem,NCBI_ID,Compound::DB00619,Gene::84959
184502,26757,Compound_Gene,MAOB,Compound,INTACT::PHYSICAL ASSOCIATION::Compound:Gene,Gene,DRKG,DB01037,monoamine oxidase B,ENSG00000069535,Pubchem,NCBI_ID,Compound::DB01037,Gene::4129


In [ ]:
save(df_Compound_Gene, 'DRKG_ChemicalEntity_Gene.csv')


---
## Dropped relation types

In [ ]:
# The following relation types are intentionally not processed:
# df_Pharmacologic_Class_Compound — entity IDs not available in standard ontologies
# df_Compound_Atc                 — ATC codes not in project scope
# df_Gene_Tax                     — taxonomy edges not in project scope
# df_Disease_Symptom              — processing logic was fully commented out in original

print("Dropped: Pharmacologic_Class_Compound, Compound_Atc, Gene_Tax, Disease_Symptom")

---
## Summary — all output files

In [ ]:
import glob as glob_mod

print(f"Output files under: {OUT_PATH}\n")
cols = ["Relation","Head_type","Tail_type","Source","KG_Source","Head_ID_IS","Tail_ID_IS"]
df_list = []

for file_path in sorted(glob_mod.glob(os.path.join(OUT_PATH, '*.csv'))):
    try:
        total_df = pd.read_csv(file_path)
        total_rows = len(total_df)
        temp_df = total_df.head(5)
        available = [c for c in cols if c in temp_df.columns]
        row = {'Filename': os.path.basename(file_path), 'no. of triples': total_rows}
        for c in available:
            row[c] = set(temp_df[c].dropna().tolist())
        df_list.append(pd.DataFrame([row]))
    except Exception as e:
        print(f"Error reading {file_path}: {e}")

combined_df = pd.concat(df_list, ignore_index=True)
print(f"Total files: {len(combined_df)}")
combined_df

Output files under: /storage/Arushi/090526_EvoAge/kg_formation/processed_data/DRKG/



/tmp/ipykernel_1605209/2790902966.py:9: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  total_df = pd.read_csv(file_path)
